# LeetCode 1010 — Pairs of Songs With Total Durations Divisible by 60 bằng Python

Link đề: https://leetcode.com/problems/pairs-of-songs-with-total-durations-divisible-by-60/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
class Solution:
    def numPairsDivisibleBy60(self, time: List[int]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất**. Điểm cốt lõi: `(a + b) % 60 == 0` ⇔ `a % 60` và `b % 60` **bù nhau** (`r` và `(60 − r) % 60`); đếm số lượng theo số dư, lưu ý hai số dư "tự bù" là `0` và `30`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm các cặp chỉ số `i < j` với `(time[i] + time[j]) % 60 == 0`.
2. Nắm công cụ Python: số dư `% 60`, mảng đếm kích thước `60`, `Counter`, tổ hợp `c·(c−1)/2`, sắp xếp + hai con trỏ.
3. Đi từ **thử mọi cặp (`O(n²)`)** → **một lượt với mảng đếm số dư** → **đếm rồi nhân các cặp số dư bù nhau** → **sắp xếp số dư + hai con trỏ** → **đếm theo giá trị và liệt kê giá trị bù**.
4. Hiểu mấu chốt: cặp `(r, (60 − r) % 60)`: với `r ∉ {0, 30}` số cặp là `cnt[r] · cnt[60 − r]`; với `r ∈ {0, 30}` (tự bù) số cặp là `C(cnt[r], 2)`.
5. Tránh lỗi kinh điển: dùng `60 − r` thay vì `(60 − r) % 60` (hỏng `r = 0`), cập nhật bộ đếm **trước** khi truy vấn (đếm cả cặp với chính nó), đếm cặp có thứ tự (nhân đôi), chỉ đếm tổng **đúng bằng** `60`, công thức tổ hợp sai.

## 2. Đề bài & ví dụ

Cho danh sách các bài hát, bài thứ `i` kéo dài `time[i]` giây. Trả về **số cặp** bài hát có tổng thời lượng chia hết cho `60`. Chính thức: số cặp chỉ số `i < j` thoả `(time[i] + time[j]) % 60 == 0`.

**Ví dụ 1:** `time = [30,20,150,100,40]` → `3` (các cặp `(30,150)`, `(20,100)`, `(20,40)`).

**Ví dụ 2:** `time = [60,60,60]` → `3` (cả ba cặp có tổng `120`).

**Ràng buộc:** `1 <= time.length <= 6·10^4`; `1 <= time[i] <= 500`.

In [1]:
import itertools
import random
import time as _time
from collections import Counter
from typing import List

examples = [([30, 20, 150, 100, 40], 3), ([60, 60, 60], 3), ([1], 0), ([30, 30, 30, 30], 6)]
for t, want in examples:
    print(f"time = {t} -> {want}")

time = [30, 20, 150, 100, 40] -> 3
time = [60, 60, 60] -> 3
time = [1] -> 0
time = [30, 30, 30, 30] -> 6


## 3. Kiến thức nền

### 3.1 Chia hết và số dư

`(a + b) % 60 == 0 ⇔ (a % 60 + b % 60) % 60 == 0`. Gọi `r = a % 60`; số dư `b % 60` phải là `(60 − r) % 60` (với `r = 0` thì cần số dư `0`, không phải `60`).

In [2]:
for a in (30, 20, 150, 60, 100, 40):
    r = a % 60
    print(f"a = {a:3d}: r = {r:2d}, cần số dư bù = {(60 - r) % 60}")

a =  30: r = 30, cần số dư bù = 30
a =  20: r = 20, cần số dư bù = 40
a = 150: r = 30, cần số dư bù = 30
a =  60: r =  0, cần số dư bù = 0
a = 100: r = 40, cần số dư bù = 20
a =  40: r = 40, cần số dư bù = 20


### 3.2 Hai số dư tự bù: `0` và `30`

`r = 0` bù với `0` (`60 − 0 = 60 ≡ 0`) và `r = 30` bù với `30`. Hai số dư này cùng nhóm với chính nó nên các cặp chọn **trong nhóm** (`C(c, 2)`), không phải `c · c`.

### 3.3 Đếm theo số dư

`cnt[r]` = số bài có `time % 60 == r`. Có `60` số dư nên mảng cố định; với `n = 6·10^4` ta chỉ cần một lượt.

In [3]:
cnt = Counter(t % 60 for t in [30, 20, 150, 100, 40])
print(dict(cnt))
total = sum(c * (c - 1) // 2 for r, c in cnt.items() if r in (0, 30)) + sum(cnt[r] * cnt[60 - r] for r in range(1, 30))
print("tổng số cặp =", total)
assert total == 3

{30: 2, 20: 1, 40: 2}
tổng số cặp = 3


## 4. Ý tưởng nền tảng / trực giác

`time = [30, 20, 150, 100, 40]`. Số dư theo `60`: `30, 20, 30, 40, 40`. Nhóm theo số dư: `{20: 1, 30: 2, 40: 2}`.

- `30` tự bù: `C(2, 2) = 1` cặp (hai bài `30` và `150`);
- `20` bù `40`: `1 · 2 = 2` cặp.

Tổng `3` ✓. Thay vì xét mọi cặp, ta đếm số bài theo số dư rồi **ghép nhóm**.

## 5. Brute force — thử mọi cặp

**Ý tưởng:** hai vòng lặp `i < j`, đếm khi `(time[i] + time[j]) % 60 == 0`. `O(n²)`; với `n = 6·10^4` là `1.8·10^9` cặp — quá chậm. Dùng làm **oracle** đúng định nghĩa.

In [4]:
def pairs_brute(time):
    n, ans = len(time), 0
    for i in range(n):
        for j in range(i + 1, n):
            if (time[i] + time[j]) % 60 == 0:
                ans += 1
    return ans


for t, want in examples:
    assert pairs_brute(t) == want
print([pairs_brute(t) for t, _ in examples])

[3, 3, 0, 6]


In [5]:
# Code nộp LeetCode cho Brute force (O(n^2): quá chậm với n = 6·10^4)
class SolutionBrute:
    def numPairsDivisibleBy60(self, time: List[int]) -> int:
        ans = 0
        for i in range(len(time)):
            for j in range(i + 1, len(time)):
                if (time[i] + time[j]) % 60 == 0:
                    ans += 1
        return ans

**Độ phức tạp:** `O(n²)` thời gian, `O(1)` bộ nhớ.

**Nút thắt:** với mỗi bài ta duyệt lại mọi bài sau nó. Chỉ cần biết **đã gặp bao nhiêu bài có số dư bù**.

## 6. Bước đột phá

Giữ `cnt[r]` = số bài đã duyệt có số dư `r`. Khi gặp bài mới có số dư `r`, cộng `cnt[(60 − r) % 60]` rồi tăng `cnt[r]`. Một lượt `O(n)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** một lượt với mảng đếm số dư (đáp án chính), **Cách 3** đếm rồi nhân các nhóm bù, **Cách 4** sắp xếp số dư + hai con trỏ, **Cách 5** đếm theo giá trị và liệt kê giá trị bù.

### Cách 2 — Một lượt với mảng đếm số dư

**Giải thích từng đoạn:**

1. `cnt = [0] * 60`.
2. Với mỗi `t`: `r = t % 60`; `ans += cnt[(60 − r) % 60]` (các bài **đã duyệt** có số dư bù); `cnt[r] += 1` (**sau** khi truy vấn).

In [6]:
def pairs_one_pass(time):
    cnt = [0] * 60
    ans = 0
    for t in time:
        r = t % 60
        ans += cnt[(60 - r) % 60]            # %60: r = 0 cần số dư 0 chứ không phải 60
        cnt[r] += 1                          # cập nhật SAU khi truy vấn (không ghép bài với chính nó)
    return ans


for t, want in examples:
    assert pairs_one_pass(t) == want
print([pairs_one_pass(t) for t, _ in examples])

[3, 3, 0, 6]


In [7]:
# Code nộp LeetCode cho Cách 2
class SolutionOnePass:
    def numPairsDivisibleBy60(self, time: List[int]) -> int:
        cnt = [0] * 60
        ans = 0
        for t in time:
            r = t % 60
            ans += cnt[(60 - r) % 60]
            cnt[r] += 1
        return ans

**Độ phức tạp:** `O(n)` thời gian, `O(60)` = `O(1)` bộ nhớ.

### Cách 3 — Đếm rồi nhân các nhóm bù

**Ý tưởng:** lượt 1 đếm `cnt[r]`. Lượt 2: với `r` từ `1` tới `29`: `cnt[r] · cnt[60 − r]`; với `r ∈ {0, 30}`: `cnt[r] · (cnt[r] − 1) // 2`.

In [8]:
def pairs_two_pass(time):
    cnt = [0] * 60
    for t in time:
        cnt[t % 60] += 1
    ans = cnt[0] * (cnt[0] - 1) // 2 + cnt[30] * (cnt[30] - 1) // 2          # hai số dư tự bù
    for r in range(1, 30):
        ans += cnt[r] * cnt[60 - r]                                          # r và 60 - r khác nhau: tích
    return ans


for t, want in examples:
    assert pairs_two_pass(t) == want
print([pairs_two_pass(t) for t, _ in examples])

[3, 3, 0, 6]


In [9]:
# Code nộp LeetCode cho Cách 3
class SolutionTwoPass:
    def numPairsDivisibleBy60(self, time: List[int]) -> int:
        cnt = [0] * 60
        for t in time:
            cnt[t % 60] += 1
        ans = cnt[0] * (cnt[0] - 1) // 2 + cnt[30] * (cnt[30] - 1) // 2
        for r in range(1, 30):
            ans += cnt[r] * cnt[60 - r]
        return ans

**Độ phức tạp:** `O(n + 60)` thời gian, `O(60)` bộ nhớ.

### Cách 4 — Sắp xếp số dư + hai con trỏ

**Ý tưởng:** sắp xếp danh sách các số dư; hai con trỏ `lo`, `hi` đi vào: nếu `rem[lo] + rem[hi] < 60` tăng `lo`; `> 60` giảm `hi`; `== 60` đếm cả khối bằng nhau ở hai phía (tích số lượng; nếu hai con trỏ cùng giá trị `30` thì `C(k, 2)`); số dư `0` được xử lý riêng (`C(c0, 2)`).

In [10]:
def pairs_sorted(time):
    rem = sorted(t % 60 for t in time)
    zeros = rem.count(0)
    ans = zeros * (zeros - 1) // 2                                        # số dư 0 tự bù
    rem = [r for r in rem if r != 0]                                      # còn lại: tổng cần đúng bằng 60
    lo, hi = 0, len(rem) - 1
    while lo < hi:
        s = rem[lo] + rem[hi]
        if s < 60:
            lo += 1
        elif s > 60:
            hi -= 1
        else:
            if rem[lo] == rem[hi]:                                        # cả đoạn [lo, hi] cùng giá trị 30
                k = hi - lo + 1
                ans += k * (k - 1) // 2
                break
            a, b = rem[lo], rem[hi]
            ca = cb = 0
            while lo <= hi and rem[lo] == a:
                lo += 1
                ca += 1
            while hi >= lo and rem[hi] == b:
                hi -= 1
                cb += 1
            ans += ca * cb
    return ans


for t, want in examples:
    assert pairs_sorted(t) == want
print([pairs_sorted(t) for t, _ in examples])

[3, 3, 0, 6]


In [11]:
# Code nộp LeetCode cho Cách 4
class SolutionSorted:
    def numPairsDivisibleBy60(self, time: List[int]) -> int:
        rem = sorted(t % 60 for t in time)
        zeros = rem.count(0)
        ans = zeros * (zeros - 1) // 2
        rem = [r for r in rem if r]
        lo, hi = 0, len(rem) - 1
        while lo < hi:
            s = rem[lo] + rem[hi]
            if s < 60:
                lo += 1
            elif s > 60:
                hi -= 1
            else:
                if rem[lo] == rem[hi]:
                    k = hi - lo + 1
                    ans += k * (k - 1) // 2
                    break
                a, b = rem[lo], rem[hi]
                ca = cb = 0
                while lo <= hi and rem[lo] == a:
                    lo += 1
                    ca += 1
                while hi >= lo and rem[hi] == b:
                    hi -= 1
                    cb += 1
                ans += ca * cb
        return ans

**Độ phức tạp:** `O(n log n)` thời gian (sắp xếp), `O(n)` bộ nhớ.

### Cách 5 — Đếm theo giá trị và liệt kê giá trị bù

**Ý tưởng:** không rút gọn về số dư; `Counter` các **giá trị** đã gặp (`1..500`). Với `t` mới, các giá trị `v` thoả `(t + v) % 60 == 0` trong `[1, 500]` là `v = (−t) mod 60 + 60k`: tối đa `⌈500/60⌉ ≈ 9` ứng viên; cộng `Counter[v]`.

In [12]:
def pairs_by_value(time, vmax=500):
    seen = Counter()
    ans = 0
    for t in time:
        start = (-t) % 60 or 60                      # giá trị bù nhỏ nhất >= 1 (v ≡ -t mod 60)
        for v in range(start, vmax + 1, 60):
            ans += seen[v]
        seen[t] += 1
    return ans


for t, want in examples:
    assert pairs_by_value(t) == want
print([pairs_by_value(t) for t, _ in examples])

[3, 3, 0, 6]


In [13]:
# Code nộp LeetCode cho Cách 5
class SolutionByValue:
    def numPairsDivisibleBy60(self, time: List[int]) -> int:
        seen = Counter()
        ans = 0
        for t in time:
            for v in range((-t) % 60 or 60, 501, 60):
                ans += seen[v]
            seen[t] += 1
        return ans

**Độ phức tạp:** `O(n · 500/60)` = `O(9n)` thời gian, `O(500)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 2** với `time = [30, 20, 150, 100, 40]`:

| `t` | `r = t % 60` | số dư bù | `cnt[bù]` | `ans` | `cnt` sau |
|---|---|---|---|---|---|
| 30 | 30 | 30 | 0 | 0 | `{30:1}` |
| 20 | 20 | 40 | 0 | 0 | `{30:1, 20:1}` |
| 150 | 30 | 30 | 1 | 1 | `{30:2, 20:1}` |
| 100 | 40 | 20 | 1 | 2 | `{30:2, 20:1, 40:1}` |
| 40 | 40 | 20 | 1 | 3 | `{30:2, 20:1, 40:2}` |

Tổng `3` ✓. Hàm trace in từng bước:

In [14]:
def trace_one_pass(time):
    cnt = [0] * 60
    ans = 0
    print("time =", time)
    for t in time:
        r = t % 60
        comp = (60 - r) % 60
        add = cnt[comp]
        ans += add
        cnt[r] += 1
        print(f"  t={t:3d}: r={r:2d}, bù={comp:2d}, cnt[bù]={add} -> ans={ans}")
    return ans


assert trace_one_pass([30, 20, 150, 100, 40]) == 3
assert trace_one_pass([60, 60, 60]) == 3

time = [30, 20, 150, 100, 40]
  t= 30: r=30, bù=30, cnt[bù]=0 -> ans=0
  t= 20: r=20, bù=40, cnt[bù]=0 -> ans=0
  t=150: r=30, bù=30, cnt[bù]=1 -> ans=1
  t=100: r=40, bù=20, cnt[bù]=1 -> ans=2
  t= 40: r=40, bù=20, cnt[bù]=1 -> ans=3
time = [60, 60, 60]
  t= 60: r= 0, bù= 0, cnt[bù]=0 -> ans=0
  t= 60: r= 0, bù= 0, cnt[bù]=1 -> ans=1
  t= 60: r= 0, bù= 0, cnt[bù]=2 -> ans=3


**Trace `[60, 60, 60]`:** số dư `0, 0, 0`; số dư bù của `0` là `(60 − 0) % 60 = 0` (không phải `60`): bài 1: `0`, bài 2: `1`, bài 3: `2` ⇒ `3` cặp.

## 9. Code nộp LeetCode

Chọn **Cách 2** (một lượt, mảng đếm). Giữ nguyên `class Solution`, `numPairsDivisibleBy60(self, time)`:

In [15]:
class Solution:
    def numPairsDivisibleBy60(self, time: List[int]) -> int:
        cnt = [0] * 60                       # cnt[r]: số bài đã duyệt có thời lượng % 60 == r
        ans = 0
        for t in time:
            r = t % 60
            ans += cnt[(60 - r) % 60]        # số dư bù (r = 0 bù với 0, r = 30 bù với 30)
            cnt[r] += 1                      # cập nhật sau khi truy vấn
        return ans


print([Solution().numPairsDivisibleBy60(t) for t, _ in examples])
assert all(Solution().numPairsDivisibleBy60(t) == w for t, w in examples)

[3, 3, 0, 6]


## 10. Vì sao đúng

**(a) Điều kiện chia hết.** `60 | a + b ⇔ (a mod 60) + (b mod 60) ≡ 0 (mod 60)`. Đặt `r = a mod 60 ∈ [0, 59]`; khi đó `b mod 60 ≡ −r (mod 60)`, tức `b mod 60 = (60 − r) mod 60`: duy nhất một giá trị trong `[0, 59]`.

**(b) Một lượt (Cách 2).** Mỗi cặp `i < j` được đếm đúng một lần tại thời điểm `j` (khi xử lý bài `j`, `cnt[(60 − r_j) % 60]` là số chỉ số `i < j` với `r_i = (60 − r_j) % 60`, tức các `i` tạo cặp hợp lệ với `j`). Cập nhật `cnt[r_j]` **sau** khi cộng đảm bảo `i ≠ j` (không ghép bài với chính nó, đặc biệt khi `r_j` tự bù: `0`, `30`).

**(c) Hai lượt (Cách 3).** Nhóm các chỉ số theo số dư. Cặp hợp lệ nối nhóm `r` với nhóm `(60 − r) % 60`. Với `r ∈ [1, 29]`: nhóm `r` và nhóm `60 − r` **khác nhau** nên số cặp = `cnt[r]·cnt[60 − r]` (mỗi cặp đếm đúng một lần vì ta chỉ lấy `r < 30`). Với `r ∈ {0, 30}`: nhóm tự bù, cặp chọn **hai phần tử khác nhau trong nhóm**: `C(c, 2)`. Mọi số dư thuộc đúng một trường hợp (`r = 0`, `r = 30`, `r ∈ [1,29]`, `r ∈ [31,59]` ứng với `60 − r ∈ [1, 29]`).

**(d) Cách 4.** Sau khi tách số dư `0` (tính `C(c0, 2)`), các số dư còn lại thuộc `[1, 59]`, cặp hợp lệ khi tổng **đúng bằng `60`** (vì tổng hai số dư thuộc `[2, 118]` chia hết `60` chỉ khi bằng `60`). Hai con trỏ trên mảng đã sắp: nếu `rem[lo] + rem[hi] < 60` thì `rem[lo]` quá nhỏ với mọi giá trị còn lại (`≤ rem[hi]`): bỏ; tương tự `> 60` bỏ `rem[hi]`. Khi `= 60` đếm cả khối bằng nhau; trường hợp `rem[lo] == rem[hi]` nghĩa là cả đoạn bằng `30` ⇒ `C(k, 2)`.

**(e) Cách 5.** Với `t` cố định, các giá trị `v ∈ [1, 500]` có `(t + v) % 60 == 0` là `v ≡ −t (mod 60)`: dãy `start, start + 60, …` với `start = ((−t) mod 60)` hoặc `60` nếu bằng `0` (vì `v ≥ 1`). Đếm `Counter[v]` cộng lại.

**(f) Kiểm chứng bằng máy.** Mọi cách khớp oracle `O(n²)` trên **mọi** mảng độ dài `≤ 5` với giá trị trong tập `{1, 2, 29, 30, 31, 58, 59, 60, 61}` (nhiều số dư bù, tự bù và dư `0`), trên mảng ngẫu nhiên tới `n = 2000`, và khớp nhau tới `n = 6·10^4` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi cặp | `O(n²)` | `O(1)` | oracle `n ≲ 3000` |
| 2. Một lượt, mảng đếm | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Đếm rồi nhân nhóm | `O(n + 60)` | `O(1)` | dạng khai báo |
| 4. Sắp xếp + hai con trỏ | `O(n log n)` | `O(n)` | không dùng bảng đếm |
| 5. Đếm theo giá trị | `O(9n)` | `O(500)` | không rút gọn số dư |

Đo thực tế `n = 6·10^4`:

In [16]:
def bench(fn, arg, repeat=5):
    best = 1e9
    for _ in range(3):
        t0 = _time.perf_counter()
        for _ in range(repeat):
            fn(arg)
        best = min(best, (_time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(1010)
big = [rng.randint(1, 500) for _ in range(60000)]
for name, fn in [("one_pass", pairs_one_pass), ("two_pass", pairs_two_pass), ("sorted", pairs_sorted), ("by_value", pairs_by_value)]:
    print(f"{name:9s}: {bench(fn, big) * 1e3:7.2f} ms (n = 6e4)")
print(f"{'brute':9s}: {bench(pairs_brute, big[:1500], repeat=1) * 1e3:7.1f} ms (n = 1500)")

one_pass :    5.53 ms (n = 6e4)


two_pass :    3.31 ms (n = 6e4)


sorted   :   13.58 ms (n = 6e4)


by_value :   40.25 ms (n = 6e4)


brute    :    72.2 ms (n = 1500)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [17]:
IMPLS = {
    "one_pass": pairs_one_pass,
    "two_pass": pairs_two_pass,
    "sorted": pairs_sorted,
    "by_value": pairs_by_value,
    "brute": pairs_brute,
    "Solution": lambda t: Solution().numPairsDivisibleBy60(list(t)),
    "SolutionOnePass": lambda t: SolutionOnePass().numPairsDivisibleBy60(list(t)),
    "SolutionTwoPass": lambda t: SolutionTwoPass().numPairsDivisibleBy60(list(t)),
    "SolutionSorted": lambda t: SolutionSorted().numPairsDivisibleBy60(list(t)),
    "SolutionByValue": lambda t: SolutionByValue().numPairsDivisibleBy60(list(t)),
    "SolutionBrute": lambda t: SolutionBrute().numPairsDivisibleBy60(list(t)),
}
SLOW = {"brute", "SolutionBrute"}
fast = [n for n in IMPLS if n not in SLOW]


def check(t, want=None, slow_limit=3000):
    if want is None:
        want = pairs_brute(t) if len(t) <= slow_limit else None
    results = {}
    for name in IMPLS:
        if name in SLOW and len(t) > slow_limit:
            continue
        results[name] = IMPLS[name](t)
    assert len(set(results.values())) == 1, (t[:8], results)
    got = next(iter(results.values()))
    if want is not None:
        assert got == want, (t[:8], got, want)
    return got


for t, want in examples:
    check(t, want)
print("ví dụ đề: oracle O(n²) và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle O(n²) và 11 cài đặt đúng


### B. Edge case

- `n = 1`: `0`.
- Toàn `60` hoặc bội của `60` (số dư `0`): `n(n−1)/2`.
- Toàn `30`: `n(n−1)/2`.
- `30` và `90`, `150` (cùng số dư `30`).
- Cặp `1` và `59`, `59` và `61`, `120`.
- Không có cặp nào: `[1, 2, 3]`.
- Giá trị lớn nhất `500` (số dư `20`): `500` và `40`.
- Hai số dư bù nhau nhiều lần: nhiều `20` và nhiều `40`.

In [18]:
edge = [
    [1], [60], [60, 60], [60, 120, 180, 240], [30, 30], [30, 90, 150, 210], [1, 59], [59, 61], [1, 2, 3], [500, 40], [500, 500],
    [20, 40, 20, 40, 20], [60, 30, 30, 60], [10, 50, 70, 110, 120], [1] * 5 + [59] * 5, [59, 1, 119, 61], [30, 60, 90, 120, 150, 180],
]
for t in edge:
    check(t)
n = 400
assert Solution().numPairsDivisibleBy60([60] * n) == n * (n - 1) // 2 == pairs_brute([60] * n)
assert Solution().numPairsDivisibleBy60([30] * n) == n * (n - 1) // 2
assert Solution().numPairsDivisibleBy60([20] * 100 + [40] * 100) == 100 * 100
print(len(edge), "edge case + toàn 60 / toàn 30 / 20×100 & 40×100: đúng")

17 edge case + toàn 60 / toàn 30 / 20×100 & 40×100: đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_no_mod_complement(time):                       # 60 - r thay vì (60 - r) % 60
    cnt = [0] * 61
    ans = 0
    for t in time:
        r = t % 60
        ans += cnt[60 - r]
        cnt[r] += 1
    return ans


def bug_update_before_query(time):                     # cập nhật trước khi truy vấn (ghép với chính nó)
    cnt = [0] * 60
    ans = 0
    for t in time:
        r = t % 60
        cnt[r] += 1
        ans += cnt[(60 - r) % 60]
    return ans


def bug_ordered_pairs(time):                           # đếm cặp có thứ tự (nhân đôi)
    return 2 * pairs_one_pass(time)


def bug_key_by_value(time):                            # khoá bộ đếm theo giá trị, tra theo 60 - t
    seen = Counter()
    ans = 0
    for t in time:
        ans += seen[60 - t]
        seen[t] += 1
    return ans


def bug_formula_plus(time):                            # C(c, 2) viết nhầm c(c+1)/2 cho 0 và 30
    cnt = [0] * 60
    for t in time:
        cnt[t % 60] += 1
    ans = cnt[0] * (cnt[0] + 1) // 2 + cnt[30] * (cnt[30] + 1) // 2
    for r in range(1, 30):
        ans += cnt[r] * cnt[60 - r]
    return ans


def bug_exactly_60(time):                              # chỉ đếm tổng đúng bằng 60
    n, ans = len(time), 0
    for i in range(n):
        for j in range(i + 1, n):
            if time[i] + time[j] == 60:
                ans += 1
    return ans


def bug_double_count_range(time):                      # nhân cnt[r]*cnt[60-r] cho r = 1..59 (đếm hai lần)
    cnt = [0] * 60
    for t in time:
        cnt[t % 60] += 1
    ans = cnt[0] * (cnt[0] - 1) // 2
    for r in range(1, 60):
        ans += cnt[r] * cnt[60 - r] if r != 30 else cnt[30] * (cnt[30] - 1) // 2
    return ans


BUGS = [bug_no_mod_complement, bug_update_before_query, bug_ordered_pairs, bug_key_by_value, bug_formula_plus, bug_exactly_60, bug_double_count_range]
probe = [[30, 20, 150, 100, 40], [60, 60, 60], [30, 30], [1, 59], [60, 120], [30, 90], [20, 40], [1, 2, 3], [60], [100, 20]]
for bug in BUGS:
    found = next(((t, bug(t), pairs_brute(t)) for t in probe if bug(t) != pairs_brute(t)), None)
    assert found, bug.__name__
    print(f"{bug.__name__:24s} sai trên {found[0]}: ra {found[1]}, đúng {found[2]}")

bug_no_mod_complement    sai trên [60, 60, 60]: ra 0, đúng 3
bug_update_before_query  sai trên [30, 20, 150, 100, 40]: ra 5, đúng 3
bug_ordered_pairs        sai trên [30, 20, 150, 100, 40]: ra 6, đúng 3
bug_key_by_value         sai trên [30, 20, 150, 100, 40]: ra 1, đúng 3
bug_formula_plus         sai trên [30, 20, 150, 100, 40]: ra 5, đúng 3
bug_exactly_60           sai trên [30, 20, 150, 100, 40]: ra 1, đúng 3
bug_double_count_range   sai trên [30, 20, 150, 100, 40]: ra 5, đúng 3


### D. Vét cạn mọi mảng nhỏ

Giá trị lấy từ tập `V = {1, 2, 29, 30, 31, 58, 59, 60, 61}` (số dư `1, 2, 29, 30, 31, 58, 59, 0, 1`) — đủ nhiều quan hệ bù, tự bù và dư `0`. Mọi mảng độ dài `1..5` (`9 + 81 + 729 + 6561 + 59049 = 66 429` mảng). Oracle `pairs_brute`.

In [20]:
V = [1, 2, 29, 30, 31, 58, 59, 60, 61]
t0 = _time.perf_counter()
cnt = 0
for n in range(1, 6):
    for arr in itertools.product(V, repeat=n):
        arr = list(arr)
        want = pairs_brute(arr)
        for name in IMPLS:
            assert IMPLS[name](arr) == want, (name, arr)
        cnt += 1
print(cnt, "mảng (n ≤ 5, giá trị trong V):", len(IMPLS), "cài đặt khớp oracle O(n²);", round(_time.perf_counter() - t0, 1), "s")
assert cnt == 66429

66429 mảng (n ≤ 5, giá trị trong V): 11 cài đặt khớp oracle O(n²); 1.7 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Khớp brute (với `n ≤ 2000`); (2) không phụ thuộc thứ tự (xáo trộn); (3) cộng bội của `60` vào phần tử **không đổi** đáp án (giá trị có thể vượt 500: ta dùng Cách 2/3/4, không dùng Cách 5 cho giới hạn `500`); (4) `0 ≤ đáp án ≤ n(n−1)/2`; (5) ghép hai mảng: đáp án của `A + B` = đáp án `A` + đáp án `B` + số cặp chéo (kiểm bằng đếm trực tiếp); (6) thêm một phần tử `x` làm tăng đáp án đúng bằng số phần tử có số dư bù.

In [21]:
rng = random.Random(20240110)
t0 = _time.perf_counter()
for it in range(400):
    n = rng.choice([1, 2, 5, 20, 100, 500, 2000])
    t = [rng.randint(1, 500) for _ in range(n)]
    if it % 3 == 0:
        t = [rng.choice([30, 60, 90, 120, 150, 20, 40, 100, 200, 10, 50]) for _ in range(n)]      # nhiều số dư tự bù/bù
    want = pairs_brute(t)                                                                           # (1)
    for name in fast:
        assert IMPLS[name](t) == want, (name, it)
    sh = t[:]
    rng.shuffle(sh)
    assert Solution().numPairsDivisibleBy60(sh) == want                                             # (2)
    shifted = [x + 60 * rng.randint(0, 5) for x in t]
    for name in ("one_pass", "two_pass", "sorted", "Solution"):
        assert IMPLS[name](shifted) == want                                                         # (3)
    assert 0 <= want <= n * (n - 1) // 2                                                            # (4)
    m = rng.randint(1, 100)
    other = [rng.randint(1, 500) for _ in range(m)]
    cross = sum(1 for a in t for b in other if (a + b) % 60 == 0)
    assert Solution().numPairsDivisibleBy60(t + other) == want + pairs_brute(other) + cross        # (5)
    x = rng.randint(1, 500)
    assert Solution().numPairsDivisibleBy60(t + [x]) == want + sum(1 for a in t if (a + x) % 60 == 0)   # (6)
print("400 mảng ngẫu nhiên (n ≤ 2000): 6 tính chất đúng;", round(_time.perf_counter() - t0, 1), "s")

400 mảng ngẫu nhiên (n ≤ 2000): 6 tính chất đúng; 10.7 s


### F. Giới hạn

`n = 6·10^4`: toàn `60`, toàn `30`, ngẫu nhiên, hai nhóm bù nhau `20/40`, giá trị lớn `500`; các cách `O(n)`/`O(n log n)` khớp nhau (brute bỏ qua).

In [22]:
t0 = _time.perf_counter()
n = 60000
rng = random.Random(10101)
cases = [
    ([60] * n, n * (n - 1) // 2), ([30] * n, n * (n - 1) // 2), ([20] * (n // 2) + [40] * (n // 2), (n // 2) ** 2),
    ([500] * n, None), ([rng.randint(1, 500) for _ in range(n)], None), ([rng.choice([30, 60]) for _ in range(n)], None),
]
for arr, want in cases:
    results = {name: IMPLS[name](arr) for name in fast}
    assert len(set(results.values())) == 1, results
    if want is not None:
        assert results["Solution"] == want
    print(f"  đáp án = {results['Solution']}")
# 500 % 60 = 20: toàn 500 => không bao giờ bù (20 + 20 = 40), đáp án 0
assert Solution().numPairsDivisibleBy60([500] * n) == 0
print("n = 6·10^4: mọi cài đặt nhanh khớp nhau;", round(_time.perf_counter() - t0, 2), "s")

  đáp án = 1799970000


  đáp án = 1799970000


  đáp án = 900000000


  đáp án = 0


  đáp án = 29963260


  đáp án = 899970324
n = 6·10^4: mọi cài đặt nhanh khớp nhau; 0.92 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: số dư bù `60 − r` không lấy `% 60`

In [23]:
t = [60, 60, 60]
print("bug:", bug_no_mod_complement(t), "| đúng:", Solution().numPairsDivisibleBy60(t))
assert bug_no_mod_complement(t) == 0 and Solution().numPairsDivisibleBy60(t) == 3

bug: 0 | đúng: 3


**Vì sao sai:** với `r = 0` số dư bù là `0`, không phải `60` (không tồn tại số dư `60`). **Đúng:** `(60 − r) % 60`.

### Lỗi 2: cập nhật bộ đếm trước khi truy vấn

In [24]:
t = [30, 30]
print("bug:", bug_update_before_query(t), "| đúng:", Solution().numPairsDivisibleBy60(t))
assert bug_update_before_query(t) == 3 and Solution().numPairsDivisibleBy60(t) == 1

bug: 3 | đúng: 1


**Vì sao sai:** với số dư tự bù (`0`, `30`) bộ đếm đã chứa chính bài đang xét ⇒ ghép nó với chính nó. **Đúng:** truy vấn trước, cập nhật sau.

### Lỗi 3: đếm cặp có thứ tự

In [25]:
t = [30, 20, 150, 100, 40]
print("bug:", bug_ordered_pairs(t), "| đúng:", Solution().numPairsDivisibleBy60(t))
assert bug_ordered_pairs(t) == 6

bug: 6 | đúng: 3


**Vì sao sai:** đề đếm cặp `i < j` (không phân biệt thứ tự). Duyệt một lượt và chỉ nhìn phía **trước** đảm bảo mỗi cặp đúng một lần.

### Lỗi 4: khoá theo giá trị, tra `60 − t`

In [26]:
t = [100, 20]
print("bug:", bug_key_by_value(t), "| đúng:", Solution().numPairsDivisibleBy60(t))
assert bug_key_by_value(t) == 0 and Solution().numPairsDivisibleBy60(t) == 1

bug: 0 | đúng: 1


**Vì sao sai:** thời lượng có thể lớn hơn `60`; `100 + 20 = 120` chia hết `60` nhưng `60 − 20 = 40 ≠ 100`. Phải rút gọn theo **số dư** (hoặc liệt kê mọi giá trị bù như Cách 5).

### Lỗi 5: công thức tổ hợp sai

In [27]:
t = [30, 30, 30]
print("bug:", bug_formula_plus(t), "| đúng:", Solution().numPairsDivisibleBy60(t))
assert bug_formula_plus(t) == 6 and Solution().numPairsDivisibleBy60(t) == 3

bug: 6 | đúng: 3


**Vì sao sai:** chọn 2 phần tử khác nhau từ `c` phần tử là `C(c, 2) = c(c−1)/2`.

### Lỗi 6 (bonus): chỉ đếm tổng đúng bằng 60

In [28]:
t = [60, 120]
print("bug:", bug_exactly_60(t), "| đúng:", Solution().numPairsDivisibleBy60(t))
assert bug_exactly_60(t) == 0 and Solution().numPairsDivisibleBy60(t) == 1

bug: 0 | đúng: 1


**Vì sao sai:** "chia hết cho 60" gồm cả `120`, `180`, ...; chứ không chỉ `60`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "đếm cặp có tổng chia hết cho `k`" → rút gọn theo **số dư** và ghép các nhóm bù `(r, (k − r) % k)`; một lượt với bảng đếm.

**Mẹo:**

1. Số dư bù `(k − r) % k` (đừng quên `% k`).
2. Hai số dư tự bù: `0` và `k/2` (nếu `k` chẵn) dùng `C(c, 2)`.
3. Một lượt: truy vấn trước, cập nhật sau.
4. Giá trị nhỏ (`≤ 500`) cho phép đếm trực tiếp theo giá trị.
5. Oracle `O(n²)` theo định nghĩa.
6. Tương tự cho tổng ba (dùng tích chập các số dư).

**Họ bài cùng pattern:** Two Sum (1), Subarray Sums Divisible by K (974), Count Pairs With XOR..., Check If Array Pairs Are Divisible by k (1497), Max Number of K-Sum Pairs (1679), Count Number of Pairs With Absolute Difference K (2006).

## 15. Biến thể

1. **Modulus bất kỳ `k`** (tổng quát hoá).
2. **Đếm bộ ba** `i < j < l` với tổng chia hết cho `60`.
3. **Liệt kê các cặp** (chỉ số) thay vì chỉ đếm.

Cài đặt và đối chiếu brute:

In [29]:
def pairs_mod_k(time, k):
    cnt = [0] * k
    ans = 0
    for t in time:
        r = t % k
        ans += cnt[(k - r) % k]
        cnt[r] += 1
    return ans


def triples_mod_k(time, k):
    """Biến thể 2: số bộ ba chỉ số i < j < l với tổng chia hết cho k, bằng đếm số dư (O(n + k²))."""
    cnt = [0] * k
    for t in time:
        cnt[t % k] += 1
    total = 0
    for a in range(k):
        for b in range(a, k):
            c = (-a - b) % k
            if c < b:
                continue                                    # chỉ đếm bộ (a <= b <= c)
            if a == b == c:
                total += cnt[a] * (cnt[a] - 1) * (cnt[a] - 2) // 6
            elif a == b:
                total += cnt[a] * (cnt[a] - 1) // 2 * cnt[c]
            elif b == c:
                total += cnt[a] * (cnt[b] * (cnt[b] - 1) // 2)
            else:
                total += cnt[a] * cnt[b] * cnt[c]
    return total


def list_pairs(time):
    return [(i, j) for i in range(len(time)) for j in range(i + 1, len(time)) if (time[i] + time[j]) % 60 == 0]


def list_pairs_fast(time):
    by_rem = {}
    out = []
    for j, t in enumerate(time):
        r = t % 60
        for i in by_rem.get((60 - r) % 60, []):
            out.append((i, j))
        by_rem.setdefault(r, []).append(j)
    return out


rng = random.Random(15)
for _ in range(400):
    n = rng.randint(1, 14)
    k = rng.choice([2, 3, 5, 60, 7])
    t = [rng.randint(1, 120) for _ in range(n)]
    assert pairs_mod_k(t, k) == sum(1 for i in range(n) for j in range(i + 1, n) if (t[i] + t[j]) % k == 0)                      # (1)
    assert triples_mod_k(t, k) == sum(1 for i, j, l in itertools.combinations(range(n), 3) if (t[i] + t[j] + t[l]) % k == 0)       # (2)
    assert sorted(list_pairs_fast(t)) == list_pairs(t)                                                                              # (3)
assert pairs_mod_k([30, 20, 150, 100, 40], 60) == 3
print("modulus bất kỳ, bộ ba, liệt kê cặp: khớp vét cạn")

modulus bất kỳ, bộ ba, liệt kê cặp: khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cặp | định nghĩa | `O(n²)` | `O(1)` | oracle |
| 2. Một lượt, mảng đếm | truy vấn rồi cập nhật | `O(n)` | `O(1)` | **đáp án chính** |
| 3. Đếm rồi nhân nhóm | `cnt[r]·cnt[60−r]` | `O(n + 60)` | `O(1)` | dạng khai báo |
| 4. Sắp xếp + hai con trỏ | tổng số dư = 60 | `O(n log n)` | `O(n)` | không bảng đếm |
| 5. Đếm theo giá trị | liệt kê giá trị bù | `O(9n)` | `O(500)` | giá trị nhỏ |

Mọi cách khớp oracle `O(n²)` trên `66 429` mảng nhỏ và khớp nhau tới `n = 6·10^4`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(Solution().numPairsDivisibleBy60([30, 20, 150, 100, 40]), Solution().numPairsDivisibleBy60([60, 60, 60]), Solution().numPairsDivisibleBy60([1]),
      Solution().numPairsDivisibleBy60([30, 30, 30, 30]), Solution().numPairsDivisibleBy60([1, 59, 61, 119]))

3 3 0 6 4


### Bài 2 (tính tay)

`time = [10, 50, 70, 110, 120]`. (a) Số dư theo `60`? (b) Đếm theo nhóm. (c) Đáp án? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện cách một lượt:

In [31]:
def one_pass_todo(time):
    cnt = [0] * 60
    ans = 0
    for t in time:
        r = t % 60
        pass            # TODO 1: ans += số dư bù (nhớ % 60)
        pass            # TODO 2: tăng cnt[r] (sau khi truy vấn)
    return ans


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(one_pass_todo(t) == w for t, w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **đếm hai lượt** (`cnt[r]·cnt[60−r]` và `C(c, 2)` cho `0`, `30`):

In [32]:
class MySolution:
    def numPairsDivisibleBy60(self, time):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().numPairsDivisibleBy60(list(t)) == w for t, w in examples)

### Bài 5 (mini challenge): bộ ba chia hết cho 60

Viết `triples_ans(time)` đếm số bộ ba `i < j < l` có `(time[i] + time[j] + time[l]) % 60 == 0`. Gợi ý: đếm theo số dư rồi xét `a ≤ b ≤ c` với `c = (−a − b) mod 60`; chú ý các trường hợp `a = b = c`, `a = b`, `b = c`. Kiểm bằng brute `O(n³)`.

In [33]:
def triples_ans(time):
    pass            # TODO


# Khi làm xong:
# assert triples_ans([20, 20, 20]) == 1 and triples_ans([30, 30, 60]) == 1

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: số dư theo `60` và ghép bù.
- Mức 2: `[30,30,30,30]`: số dư `30` tự bù ⇒ `C(4, 2)`.
- Mức 3: với `[1,59,61,119]` số dư là `1, 59, 1, 59` ⇒ `2·2 = 4`; kết quả `3 3 0 6 4`.

**Bài 2**

- Mức 1: `10, 50, 10, 50, 0`.
- Mức 2: nhóm `{10: 2, 50: 2, 0: 1}`.
- Mức 3: `10` bù `50`: `2·2 = 4`; số dư `0`: `C(1, 2) = 0` ⇒ `4`.

**Bài 3**

- Mức 1: `ans += cnt[(60 - r) % 60]`.
- Mức 2: `cnt[r] += 1` sau đó.
- Mức 3: trả `ans`.

**Bài 4**

- Mức 1: lượt 1 đếm `cnt`.
- Mức 2: `ans = C(cnt[0], 2) + C(cnt[30], 2)`.
- Mức 3: `for r in range(1, 30): ans += cnt[r] * cnt[60 - r]`.

**Bài 5**

- Mức 1: đếm `cnt` theo số dư.
- Mức 2: duyệt `a ≤ b`, `c = (-a - b) % 60`, bỏ qua `c < b`.
- Mức 3: `a == b == c`: `C(c, 3)`; `a == b < c`: `C(cnt[a], 2) · cnt[c]`; `a < b == c`: `cnt[a] · C(cnt[b], 2)`; ngược lại tích ba.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
vals = (Solution().numPairsDivisibleBy60([30, 20, 150, 100, 40]), Solution().numPairsDivisibleBy60([60, 60, 60]), Solution().numPairsDivisibleBy60([1]),
        Solution().numPairsDivisibleBy60([30, 30, 30, 30]), Solution().numPairsDivisibleBy60([1, 59, 61, 119]))
print(vals)
assert vals == (3, 3, 0, 6, 4)
print("Bài 1 OK")

(3, 3, 0, 6, 4)
Bài 1 OK


### Bài 2

In [35]:
t = [10, 50, 70, 110, 120]
rem = [x % 60 for x in t]
print(rem, Counter(rem))
assert rem == [10, 50, 10, 50, 0] and Solution().numPairsDivisibleBy60(t) == 4 == pairs_brute(t)
print("Bài 2 OK: 2·2 + C(1,2) = 4")

[10, 50, 10, 50, 0] Counter({10: 2, 50: 2, 0: 1})
Bài 2 OK: 2·2 + C(1,2) = 4


### Bài 3

In [36]:
def one_pass_answer(time):
    cnt = [0] * 60
    ans = 0
    for t in time:
        r = t % 60
        ans += cnt[(60 - r) % 60]
        cnt[r] += 1
    return ans


assert all(one_pass_answer(t) == w for t, w in examples)
rng = random.Random(3)
for _ in range(300):
    t = [rng.randint(1, 500) for _ in range(rng.randint(1, 40))]
    assert one_pass_answer(t) == pairs_brute(t)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def numPairsDivisibleBy60(self, time):
        cnt = [0] * 60
        for t in time:
            cnt[t % 60] += 1
        ans = cnt[0] * (cnt[0] - 1) // 2 + cnt[30] * (cnt[30] - 1) // 2
        for r in range(1, 30):
            ans += cnt[r] * cnt[60 - r]
        return ans


assert all(MySolutionAnswer().numPairsDivisibleBy60(list(t)) == w for t, w in examples)
rng = random.Random(4)
for _ in range(300):
    t = [rng.randint(1, 500) for _ in range(rng.randint(1, 40))]
    assert MySolutionAnswer().numPairsDivisibleBy60(t) == pairs_brute(t)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def triples_ans(time):
    return triples_mod_k(time, 60)


def triples_brute(time):
    return sum(1 for i, j, l in itertools.combinations(range(len(time)), 3) if (time[i] + time[j] + time[l]) % 60 == 0)


assert triples_ans([20, 20, 20]) == 1 and triples_ans([30, 30, 60]) == 1
rng = random.Random(5)
for _ in range(300):
    t = [rng.choice([rng.randint(1, 120), 20, 30, 40, 60]) for _ in range(rng.randint(1, 16))]
    assert triples_ans(t) == triples_brute(t)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đếm cặp có tổng chia hết cho `k` → bảng đếm số dư, một lượt):**

```python
class Solution:
    def numPairsDivisibleBy60(self, time: List[int]) -> int:
        cnt = [0] * 60
        ans = 0
        for t in time:
            r = t % 60
            ans += cnt[(60 - r) % 60]
            cnt[r] += 1
        return ans
```

**Checklist trước khi submit:**

- [ ] Số dư bù `(60 − r) % 60` (có `% 60`)?
- [ ] Truy vấn **trước**, cập nhật bộ đếm **sau**?
- [ ] Mỗi cặp `i < j` đếm đúng một lần?
- [ ] Hai số dư tự bù (`0`, `30`) dùng `C(c, 2)` (nếu đếm hai lượt)?
- [ ] Đã thử `[30,20,150,100,40] → 3`, `[60,60,60] → 3`, `[1]`, toàn `30`?
- [ ] Biết đáp án có thể lớn (`~1.8·10^9`) — Python int không tràn?
- [ ] Đúng tên `class Solution`, `numPairsDivisibleBy60(self, time)`, không in debug.